# NB2: Shared RAG Memory & Theory of Mind

**2-minute intro script:** Sequential collaboration works until context fades. The PM may define a crucial constraint, but by the time the Coder or QA acts, that detail can be buried or lost. Shared memory fixes this by giving agents a governed place to write and retrieve commitments. Theory of Mind appears when Agent B asks, "What did Agent A know that should influence my decision?" In this notebook, the PM records a hidden storage constraint, the Coder discovers it through authorized memory search, and restricted secrets stay hidden.

In [ ]:
from enum import Enum
from typing import List, Set
from uuid import uuid4
from pydantic import BaseModel, ConfigDict, Field

class Role(str, Enum):
    PM = "pm"
    CODER = "coder"
    SECURITY = "security"

class Sensitivity(str, Enum):
    PUBLIC = "public"
    CONFIDENTIAL = "confidential"
    RESTRICTED = "restricted"

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class MemoryRecord(StrictModel):
    """Theory-of-Mind memory with visibility controls."""
    author: Role
    visible_to: Set[Role]
    sensitivity: Sensitivity
    tags: Set[str]
    text: str
    record_id: str = Field(default_factory=lambda: f"mem_{uuid4().hex[:8]}")

class SharedMemory:
    """Production-style shared memory with authorization."""

    def __init__(self):
        self.records: List[MemoryRecord] = []

    def add(self, record: MemoryRecord) -> str:
        self.records.append(record)
        return record.record_id

    def search(self, query: str, requester: Role) -> List[MemoryRecord]:
        query_lower = query.lower()
        results = []

        for record in self.records:
            if requester not in record.visible_to:
                continue
            if record.sensitivity == Sensitivity.RESTRICTED and requester != Role.SECURITY:
                continue

            text_match = query_lower in record.text.lower()
            tag_match = any(query_lower in tag.lower() for tag in record.tags)

            if text_match or tag_match:
                results.append(record)

        return results

In [ ]:
class DesignPlan(StrictModel):
    storage_choice: str
    rationale: str
    risks: List[str] = Field(default_factory=list)

def coder_design_from_memory(task: str, memory: SharedMemory) -> DesignPlan:
    records = memory.search("storage", Role.CODER)
    storage_choice = "unknown"
    rationale = "No architecture memory was found."

    for record in records:
        if "postgresql" in record.text.lower():
            storage_choice = "PostgreSQL"
            rationale = f"Discovered PM constraint in {record.record_id}: {record.text}"

    return DesignPlan(
        storage_choice=storage_choice,
        rationale=rationale,
        risks=["Validate migrations", "Confirm backup and recovery policy"],
    )

In [ ]:
def demo_theory_of_mind():
    memory = SharedMemory()

    memory.add(MemoryRecord(
        author=Role.PM,
        visible_to={Role.CODER, Role.SECURITY},
        sensitivity=Sensitivity.CONFIDENTIAL,
        tags={"architecture", "storage", "constraint"},
        text="Use PostgreSQL for persistent storage. Do not use SQLite.",
    ))

    memory.add(MemoryRecord(
        author=Role.SECURITY,
        visible_to={Role.SECURITY},
        sensitivity=Sensitivity.RESTRICTED,
        tags={"credentials"},
        text="Production DB password: super_secret_123",
    ))

    print("=== Coder searching for 'storage' ===")
    for record in memory.search("storage", Role.CODER):
        print(f"Found: {record.text}")
        print(f"Tags: {sorted(record.tags)}")
        print(f"Sensitivity: {record.sensitivity.value}")

    print("\n=== Coder trying to access 'credentials' ===")
    print(f"Results: {len(memory.search('credentials', Role.CODER))} (should be 0)")

    print("\n=== Security searching for 'credentials' ===")
    print(f"Results: {len(memory.search('credentials', Role.SECURITY))} (should be 1)")

    print("\n=== Coder design plan ===")
    print(coder_design_from_memory("Implement user persistence layer.", memory))

demo_theory_of_mind()

## Context Window Economics

Shared memory is not only about remembering facts. It is also about managing cognitive load. Production agents fail when their short-term context grows until the model loses constraints or hits a token limit. A manager agent should compress old context into durable memory while preserving commitments.

In [ ]:
class ConversationMessage(StrictModel):
    speaker: Role
    text: str

class TeamStateHistory(StrictModel):
    short_term_history: List[ConversationMessage] = Field(default_factory=list)

    def token_count(self) -> int:
        # Teaching approximation: production uses tokenizer-specific counts.
        return sum(len(message.text.split()) for message in self.short_term_history)

class ContextCompressionReport(StrictModel):
    original_tokens: int
    compressed_tokens: int
    compression_ratio: float
    token_savings: int
    summary_record_ids: List[str]
    critical_constraints_preserved: bool

class ContextSummarizer:
    """Compress short-term history into governed long-term memory."""

    def __init__(self, threshold_tokens: int = 20_000, target_records: int = 5):
        self.threshold_tokens = threshold_tokens
        self.target_records = target_records

    def compress_if_needed(
        self,
        state: TeamStateHistory,
        memory: SharedMemory,
    ) -> ContextCompressionReport | None:
        original_tokens = state.token_count()
        if original_tokens <= self.threshold_tokens:
            return None

        critical_constraints = [
            message.text
            for message in state.short_term_history
            if "CRITICAL_CONSTRAINT" in message.text
        ]

        chunk_size = max(1, len(state.short_term_history) // self.target_records)
        summary_ids: List[str] = []
        compressed_tokens = 0

        for index in range(self.target_records):
            start = index * chunk_size
            end = len(state.short_term_history) if index == self.target_records - 1 else (index + 1) * chunk_size
            chunk = state.short_term_history[start:end]
            if not chunk:
                continue

            speakers = sorted({message.speaker.value for message in chunk})
            local_constraints = [
                message.text
                for message in chunk
                if "CRITICAL_CONSTRAINT" in message.text
            ]
            constraint_text = " ".join(local_constraints or critical_constraints[:1])
            summary_text = (
                f"Compressed context chunk {index + 1}: speakers={speakers}. "
                f"{constraint_text}"
            )
            compressed_tokens += len(summary_text.split())
            summary_ids.append(
                memory.add(
                    MemoryRecord(
                        author=Role.PM,
                        visible_to={Role.CODER, Role.SECURITY},
                        sensitivity=Sensitivity.CONFIDENTIAL,
                        tags={"summary", "compressed_context", "constraint"},
                        text=summary_text,
                    )
                )
            )

        state.short_term_history.clear()
        preserved = bool(memory.search("PostgreSQL", Role.CODER))
        return ContextCompressionReport(
            original_tokens=original_tokens,
            compressed_tokens=compressed_tokens,
            compression_ratio=round(compressed_tokens / original_tokens, 4),
            token_savings=original_tokens - compressed_tokens,
            summary_record_ids=summary_ids,
            critical_constraints_preserved=preserved,
        )

def demo_context_compression():
    memory = SharedMemory()
    state = TeamStateHistory()
    filler = "implementation detail " * 260

    for index in range(100):
        text = f"Message {index}: {filler}"
        if index == 42:
            text += " CRITICAL_CONSTRAINT: Use PostgreSQL for persistent storage."
        state.short_term_history.append(
            ConversationMessage(
                speaker=Role.PM if index % 3 == 0 else Role.CODER,
                text=text,
            )
        )

    report = ContextSummarizer().compress_if_needed(state, memory)
    print(report.model_dump_json(indent=2))
    print(f"Short-term messages remaining: {len(state.short_term_history)}")
    print(f"PostgreSQL summaries visible to coder: {len(memory.search('PostgreSQL', Role.CODER))}")

    assert report is not None
    assert report.original_tokens > 20_000
    assert len(report.summary_record_ids) == 5
    assert state.short_term_history == []
    assert report.critical_constraints_preserved is True
    assert len(memory.search("PostgreSQL", Role.CODER)) >= 1

demo_context_compression()

## 🧪 Exercises: Building the Institutional Brain

**The Story:** You are the Chief Architect. Your team is growing. The PM just told the Coder to use PostgreSQL, but the QA team is testing against SQLite because they didn't get the memo. Meanwhile, your context window is filling up with 10,000 messages of "sounds good" and "will fix". How do we give agents a shared, secure memory without blowing up our token budget?

**Your Mission:**
1. **The Need-to-Know Basis:** Add a `QA` role to the system. Let QA see acceptance criteria and test results, but strictly forbid them from seeing the `credentials` memory records.
2. **The Ghost of Decisions Past:** Add a memory record with stale architecture guidance (e.g., "Use MySQL"). Use tags to filter it out so the Coder only sees the current `PostgreSQL` constraint.
3. **The Fuzzy Search:** Change the `search` function so it returns both exact tag hits and text-token hits. This mimics how real RAG systems retrieve context.
4. **The Inner Monologue:** Add a `source_agent_thought` field to the `MemoryRecord`. Discuss: Should this field be visible to other agents, or should it remain private to the author?
5. **Scaling the Brain:** Replace the in-memory Python list with a real vector store like ChromaDB or FAISS, while preserving the exact same authorization and visibility rules.
6. **The Forgetting Curve:** Change the compression threshold in the `ContextSummarizer`. Measure how many summaries are produced and verify that the critical `PostgreSQL` constraint survives the compression.

**The Takeaway:** Shared memory does not mean *open* memory. A managed team enforces need-to-know, and compresses the past to focus on the present.

In [ ]:
# ==========================================
# LIVE LLM EXECUTION (Optional)
# ==========================================
# The cells above run offline using deterministic mocks.
# To see a real LLM generate output constrained by this schema:
#
#   pip install openai instructor
#   export OPENAI_API_KEY="..."
#
# Keep this False for workshops unless learners have API keys.
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    import os
    import instructor
    from openai import OpenAI

    client = instructor.from_openai(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))
    model_name = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

    live_result = client.chat.completions.create(
        model=model_name,
        response_model=MemoryRecord,
        messages=[
            {
                "role": "user",
                "content": 'Create a governed memory record for a PM constraint: use PostgreSQL for customer persistence. Make it visible to coder and security.',
            }
        ],
    )
    print(live_result.model_dump_json(indent=2))